<!-- markdownlint-disable MD033 MD041 -->

# <ins>Informe de Trabajo Práctico: Recuperación de Información en la Web</ins>
## Estructuras de Datos 

---
### Grupo: Recurso.py
*<ins>Integrantes:</ins>*
- Abril Reinaga
- Delfina Emma Basso Natale
- Matías Di Bernardo

---

Se solicitó desarrollar un programa en Python que procese información de distintas fuentes de la Web, con tres distintas técnicas de recuperación de datos, y lo guarde en archivos CSV.

El objetivo de este programa es que los integrantes del grupo apliquen los conocimientos aprendidos en Estructuras de Datos, específicamente las técnicas de recuperación de información: **APIs**, **Web Scraping** y **RSS**. El programa resultante ha sido implementado con el lenguaje de programación _Python_.

Como fuentes para la recopilación de datos se han utilizado `(añadir las de API y Web Scraping)`, Investing, El Economista y Wall Street Journal.

In [1]:
#Bloque para API, añadir más bloques para muestra de código de ser necesario!

In [ ]:
#Bloque para Web Scraping, añadir más bloques para muestra de código de ser necesario!
boton_ubicacion = (By.XPATH, "//div[@class='pagination float-end col-8 d-flex align-items-center small mb-3 justify-content-end pe-0']//span[@class='btn btn-sm btn-primary me-1 cursor-pointer pl-n']") 
boton = WebDriverWait(driver, 10).until(EC.element_to_be_clickable(boton_ubicacion))

boton.click()
  
WebDriverWait(driver, 10).until(EC.visibility_of_element_located(By.XPATH, "//div[@class='pagination float-end col-8 d-flex align-items-center small mb-3 justify-content-end pe-0']//span[@class='btn btn-sm bg-white current me-1 border-0 rounded border_dl']"))

soup = BeautifulSoup(driver.page_source, "html.parser")

time.sleep(retraso)


---
## RSS Feeds

La estructura para recuperar información de los Feeds RSS está implementada a partir de la librería para Python `feedparser`, la cual facilita la organización de los datos segun sus etiquetas `HTML`, guardandolo en diccionarios de formato etiqueta/contenido. Se pidió extraer, de cada noticia, el título, la descripción, la fecha de publicación y país asociado a la noticia. Toda esta información, luego, se guarda en un archivo de formato CSV.

Para _parsear_ las páginas web, el módulo _parser_ se dividió en tres funciones principales: `clean_hmtl_tags(html_txt)`, `get_country_name(entry)` y `process_rss_feeds(url, path)`:


- `clean_html_tags(html_txt)`: Esta función recibe un texto en formato **HTML** para eliminar cualquier cadena de texto que coincida con la de una etiqueta, con o sin atributos, dada una expresión regular y la devuelve.

In [10]:
import re

def clean_html_tags(html_txt) -> str:
    if not html_txt:
        return ""
    txt = html_txt
    txt = re.sub(r'<!\[CDATA\[|\]\]>', '', txt)
    txt = re.sub(r'<img[^>]*>', '', txt)
    txt = re.sub(r'<.*?>', '', txt)
    
    return txt

texto_con_etiquetas = '<p>Hola <b>mundo</b>, esta es una <a href="#">prueba</a>.</p>]]>'
texto_sin_etiquetas = clean_html_tags(texto_con_etiquetas)

print(f"Antes: {texto_con_etiquetas}")
print(f"Después: {texto_sin_etiquetas}")

Antes: <p>Hola <b>mundo</b>, esta es una <a href="#">prueba</a>.</p>]]>
Después: Hola mundo, esta es una prueba.


- `get_country_name(entry)`: Esta función recibe una entrada del feed en formato etiqueta/contenido y devuelve el país asociado al título o descripción de esa noticia. Convierte tanto el título como la descripción en una sola cadena para buscar cualquier nombre de país o gentilicio que aparezca en la noticia. Si lo encuentra, devuelve el nombre del país. Si no, devuelve "País no encontrado".

In [13]:
#Ejemplo de diccionario de gentilicios
countries_aliases = {"Reino Unido": ["uk", "united kingdom", "british", "británico", "británica"]}

def get_country_name(entry)->str:
    txt_title = entry.get("title", "")
    txt_description = entry.get("description", "")
    txt = f"{clean_html_tags(txt_title).lower()} {clean_html_tags(txt_description).lower()}"
    
    for country, aliases in countries_aliases.items():
        for alias in aliases:
            pattern = r'\b' + re.escape(alias) + r"(?:['’]s)?\b" #En caso de que al nombre lo acompañe un 's o ’s
            if re.search(pattern, txt):
                return country  
                
    return "Country not specified"

entrada_de_prueba = {
        "title": "<title>La economía británica se estabiliza</title>",
        "description": ""
    }
pais_de_noticia = get_country_name(entrada_de_prueba)

print(f"Texto de prueba: {entrada_de_prueba["title"]}")
print(f"País asociado: {pais_de_noticia}")

Texto de prueba: <title>La economía británica se estabiliza</title>
País asociado: Reino Unido


- `process_rss_feeds(url, path)`: Esta función es el corazón de la implementación, ya que dados un URL del feed y una dirección a una carpeta, extrae la información solicitada, la guarda en un diccionario y agrega en una lista de diccionarios de entradas. Se utiliza la librería `pandas` para organizar cada diccionario de la lista en formato CSV, usando las claves como encabezado del archivo.

<sub>_Nota_: Como el bloque de código de esta función es el más extenso de los tres, se cortó la parte más importante.</sub>

In [17]:
import feedparser

url = ""
all_articles = []

feed = feedparser.parse(url)
        
for entry in feed.entries:
    title = entry.get("title", "Untitled")
    pub_date = entry.get("pubDate") or entry.get("published") or "Publication date not found"
    summary = clean_html_tags(entry.get("description", "Description not found"))
    country_name = get_country_name(entry)
            
    article_data = {
        "title": title,
        "publication_date": pub_date,
        "summary": summary,
        "country": country_name
    }
    all_articles.append(article_data)